# Cypher analyst workbook

Run all cells to: detect each test PDF's variant, run extraction + soft validation, and explore the results interactively. Every section below has its own purpose — you can run cells out of order once you've executed the **Setup** and **Run extraction** cells.

## What this notebook gives you that the HTML report doesn't

- **Live filtering / sorting / sub-setting** of the results dataframes (use pandas in any cell).
- **Per-row drill-down**: pick any PDF, see flagged rows highlighted, examine the `_issues` flags.
- **Cross-PDF metrics**: how clean rates differ by variant, which columns flag most often, etc.
- **Inline bounding-box debug images** for L3 (OCR) extractions.
- A **single source of truth** for the analyst — every change in the codebase that affects extraction shows up here on re-run.

The HTML report is for sharing / off-line viewing. This notebook is for *working*.

---
## Setup

Imports the project, sets pandas display, points at the test corpus.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from collections import Counter
from IPython.display import display, Image, Markdown

from sheet_types import occm

TEST_DIR    = ROOT / 'research' / 'test_pdfs'
RESULTS_DIR = ROOT / 'research' / 'results' / 'by_pdf'

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 200)

print(f'Project root: {ROOT}')
print(f'Test PDFs:    {TEST_DIR}')
print(f'Results:      {RESULTS_DIR}')

---
## Discover test PDFs and detect variants

Walks `research/test_pdfs/` and asks the OCCM router which variant each one is. Detection runs on the first ~3 pages of text; falls back to **Aeroflot** when no text layer exists.

In [ ]:
pdfs = sorted(TEST_DIR.glob('*.pdf'))
rows = []
for p in pdfs:
    rows.append({
        'pdf':     p.name,
        'size_kb': p.stat().st_size // 1024,
        'variant': occm.detect_variant(str(p)),
    })
discovery_df = pd.DataFrame(rows)
discovery_df

---
## Run extraction across the whole corpus

For each PDF, dispatch to its variant's parser (L1 for AMOS / China Eastern, L3 OCR for Aeroflot), apply rule-based normalization + soft validation, and stash the resulting dataframe in `RESULTS[pdf_name]`.

L3 is slow (OCR runs at 300 DPI) — re-running the whole corpus takes ~30s per scanned PDF. Consider commenting out the `afl_test.pdf` line if you're iterating on something else.

In [ ]:
RESULTS = {}
VARIANT_OF = {}

for p in pdfs:
    name = p.name
    variant = occm.detect_variant(str(p))
    VARIANT_OF[name] = variant
    print(f'  {name:25s}  variant={variant}')
    result = occm.extract(str(p), variant_name=variant)
    cleaned = occm.normalize_and_validate(result['records'], variant_name=variant)
    df = pd.DataFrame(cleaned)
    cols = result['columns'] + ['_issues', '_page']
    df = df[[c for c in cols if c in df.columns]]
    RESULTS[name] = df

print('\nExtraction complete. Available in RESULTS dict and VARIANT_OF dict.')

---
## Cross-PDF scoreboard

How each PDF performed end-to-end. `clean_pct` is the headline number. `imputed_ata` counts rows where the ATA chapter came from the forward-fill helper (zero today because parsers already track ATA inline).

In [ ]:
def summarize(df: pd.DataFrame, name: str, variant: str) -> dict:
    if df.empty:
        return {'pdf': name, 'variant': variant, 'rows': 0, 'clean': 0,
                'flagged': 0, 'clean_pct': 0.0, 'imputed_ata': 0}
    n = len(df)
    issues = df['_issues'].fillna('').astype(str)
    clean = (issues == '').sum()
    imputed = issues.str.contains('_imputed:ATA', na=False).sum()
    return {
        'pdf': name, 'variant': variant, 'rows': n,
        'clean': int(clean), 'flagged': int(n - clean),
        'clean_pct': round(100 * clean / n, 1),
        'imputed_ata': int(imputed),
    }

scoreboard = pd.DataFrame([summarize(RESULTS[p], p, VARIANT_OF[p]) for p in RESULTS])
scoreboard

---
## Issue-frequency breakdown

Across all PDFs, which columns are flagging most often, and with what reasons. Useful for prioritizing rule tweaks: high-frequency `bad_format` flags usually point to either a real source quality problem or a rule that needs loosening.

In [ ]:
all_issues = []
for name, df in RESULTS.items():
    if df.empty:
        continue
    for s in df['_issues'].fillna('').astype(str):
        for bit in s.split(','):
            if bit:
                col, _, reason = bit.partition(':')
                all_issues.append({'pdf': name, 'variant': VARIANT_OF[name],
                                  'column': col, 'reason': reason})

issues_df = pd.DataFrame(all_issues)
if not issues_df.empty:
    by_col_reason = (issues_df.groupby(['column', 'reason']).size()
                     .reset_index(name='count').sort_values('count', ascending=False))
    display(by_col_reason)

In [ ]:
# Same data, broken down by variant — useful when one variant has a quirk the others don't.
if not issues_df.empty:
    by_variant = (issues_df.groupby(['variant', 'column', 'reason']).size()
                  .reset_index(name='count').sort_values(['variant', 'count'], ascending=[True, False]))
    display(by_variant)

---
## Per-PDF detail view (highlighted)

Pick a PDF, see all rows with flagged rows shaded. Change `pdf_name` and re-run.

In [ ]:
def show(df: pd.DataFrame, only_flagged: bool = False, max_rows: int = 200):
    if df.empty:
        return df
    view = df[df['_issues'].fillna('').astype(str) != ''] if only_flagged else df
    view = view.head(max_rows)
    def _bg(row):
        return ['background-color: #fff5e6' if str(row.get('_issues', '')) else '' for _ in row]
    return view.style.apply(_bg, axis=1)

pdf_name = 'msn2212OCCM.pdf'   # ← edit this and re-run the cell
show(RESULTS[pdf_name])

In [ ]:
# Just the flagged rows for the same PDF — fastest way to spot what needs eyeballing.
show(RESULTS[pdf_name], only_flagged=True)

---
## Bounding-box debug overlays (L3 only)

For OCR runs, each page is rendered with words colored by which column they were assigned to. Look for misaligned colors at column boundaries — that's where the column-projection logic is mis-placing words.

In [ ]:
for name, variant in VARIANT_OF.items():
    if variant != 'Aeroflot':
        continue  # only L3 has debug overlays today
    debug_dir = RESULTS_DIR / f'{pathlib.Path(name).stem}_debug'
    if not debug_dir.exists():
        print(f'No debug overlays for {name}. Run shared.debug_render.render_debug_pages first.')
        continue
    display(Markdown(f'### {name}'))
    for img in sorted(debug_dir.glob('*.png')):
        display(Image(filename=str(img), width=900))

---
## Re-render bbox overlays on demand

Run this if you've changed the L3 extractor and want fresh overlays.

In [ ]:
from shared.debug_render import render_debug_pages
for name, variant in VARIANT_OF.items():
    if variant != 'Aeroflot':
        continue
    out_dir = RESULTS_DIR / f'{pathlib.Path(name).stem}_debug'
    paths = render_debug_pages(str(TEST_DIR / name), str(out_dir))
    print(f'  {name}: {len(paths)} pages → {out_dir}')

---
## Search across the corpus

Find every row mentioning a particular part number, FIN, description fragment, etc., regardless of which PDF it came from. Useful for cross-airframe comparison once we have many OCCMs in the corpus.

In [ ]:
def search(term: str, columns: list[str] | None = None) -> pd.DataFrame:
    """Case-insensitive substring search. `columns=None` searches every cell."""
    out = []
    for name, df in RESULTS.items():
        if df.empty:
            continue
        cols = columns or [c for c in df.columns if not c.startswith('_')]
        mask = pd.Series(False, index=df.index)
        for c in cols:
            if c in df.columns:
                mask |= df[c].astype(str).str.contains(term, case=False, na=False)
        hits = df[mask].copy()
        if not hits.empty:
            hits.insert(0, '_pdf', name)
            hits.insert(1, '_variant', VARIANT_OF[name])
            out.append(hits)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

# Example: every row mentioning 'PRESSURE' across all variants
search('PRESSURE').head(20)

In [ ]:
# Every row whose ATA chapter is 21 (Air Conditioning), grouped by variant
ata21 = pd.concat([
    RESULTS[name].assign(_pdf=name, _variant=VARIANT_OF[name])
    for name in RESULTS
    if not RESULTS[name].empty
], ignore_index=True)
ata21 = ata21[ata21['ATA'].astype(str) == '21']
ata21.groupby('_variant').size().rename('rows_in_ATA_21')

---
## Export the current state

Re-saves CSV + XLSX outputs for every PDF in this run. Use after iterating on the parsers in this notebook before the next session.

In [ ]:
for name, df in RESULTS.items():
    if df.empty:
        continue
    variant = VARIANT_OF[name]
    level = 'L3' if variant == 'Aeroflot' else 'L1'
    out = RESULTS_DIR / f"{pathlib.Path(name).stem}_{level}.csv"
    df.to_csv(out, index=False)
    df.to_excel(out.with_suffix('.xlsx'), index=False)
    print(f'  wrote {out.name} and .xlsx')

---
## Notes & TODO

- Add HT and LLP variants once example documents are available.
- Cross-reference PART_NUMBER against an authoritative PN master list (deferred — see `docs/decisions.md`).
- Once we have ~20 OCCMs in the corpus, plot clean-rate trends across variants over time.